<a href="https://colab.research.google.com/github/sbmshukla/ML-Notebook-Quick-Saved/blob/main/ANNTunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
import scikeras
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [3]:
df = pd.read_csv('Churn_Modelling.csv')

In [4]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [5]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model


In [6]:
from scikeras.wrappers import KerasClassifier

In [7]:
model = KerasClassifier(build_fn=create_model, batch_size=32, verbose=1)

In [8]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}


In [5]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)


In [6]:
grid_result = grid.fit(X_train, y_train)

AttributeError: 'super' object has no attribute '__sklearn_tags__'

In [2]:
!pip install scikeras

In [4]:
X_train

array([[1.43641014e-01, 5.04257315e-01, 8.14067943e-01, 4.42140467e-01,
        9.56175505e-02, 1.06889371e-01, 7.41280187e-01, 1.89886059e-01,
        5.63833390e-01, 1.82659530e-01],
       [1.20989083e-01, 8.56885598e-01, 1.67713955e-01, 4.73041815e-01,
        2.83665644e-01, 3.15932394e-01, 5.56336623e-01, 1.27056589e-01,
        2.93129589e-03, 6.12408945e-01],
       [4.35305297e-01, 8.04052774e-01, 9.18782818e-01, 7.94797835e-01,
        8.26230980e-01, 8.41773958e-01, 6.30970261e-01, 2.50149807e-02,
        8.02127334e-01, 9.31491970e-01],
       [1.69546549e-02, 5.16262340e-01, 2.00493770e-01, 6.61727905e-01,
        5.26015589e-01, 3.86244909e-02, 2.82016200e-01, 9.84154738e-01,
        7.34466025e-01, 8.70125561e-01],
       [9.67548390e-01, 9.06501221e-01, 7.90882106e-01, 9.24684929e-01,
        5.43986864e-01, 2.84553713e-01, 5.07028192e-01, 6.04062563e-01,
        5.60156594e-01, 6.62952270e-01],
       [2.09084721e-01, 4.56924511e-01, 6.16293119e-01, 8.46385645e-01,
   